# Séance 3 : Analyse de réseaux et analyse spatiale avec Python

**Durée :** 1 journée (7h)
**Public :** Chercheurs et chercheuses en SHS ayant suivi les Séances 1 (pandas) et 2 (fouille de texte).
**Données :** `auc.csv` (parcours d'étudiants, Séance 1) — utilisé aujourd'hui de bout en bout, pour les réseaux **et** pour la cartographie.

---

## Déroulé de la journée (indicatif, à ajuster selon le rythme du groupe)

| Horaire | Durée | Bloc | Contenu |
|---|---|---|---|
| 9h00 – 9h15 | 15 min | 0. Introduction | Réseaux et cartes : deux façons de révéler une structure cachée dans les données |
| 9h15 – 9h45 | 30 min | 1. Prise en main de NetworkX | Créer, manipuler et dessiner un premier graphe |
| 9h45 – 10h45 | 60 min | 2. Réseau d'affiliation | Universités ↔ employeurs : d'un tableau à un réseau biparti, puis projeté |
| 10h45 – 11h00 | 15 min | ☕ Pause | |
| 11h00 – 11h45 | 45 min | 3. Centralités et communautés | Qui est central ? Quels sous-groupes émergent ? |
| 11h45 – 12h45 | 60 min | 🍽️ Déjeuner | |
| 12h45 – 13h30 | 45 min | 4. Introduction à geopandas | GeoDataFrame, géométries, systèmes de coordonnées (CRS) |
| 13h30 – 14h30 | 60 min | 5. Cartographie choroplèthe | Dans quels États américains les étudiants se sont-ils formés ? |
| 14h30 – 14h45 | 15 min | ☕ Pause | |
| 14h45 – 15h30 | 45 min | 6. Cartes de points et distances | Où les étudiants se sont-ils établis en 1936 ? Quelle distance ont-ils parcourue ? |
| 15h30 – 16h15 | 45 min | 7. Réseaux + cartes | Cartographier le réseau université → ville d'installation |
| 16h15 – 16h45 | 30 min | 8. Mini-projet de synthèse | Exercice récapitulatif en autonomie |
| 16h45 – 17h00 | 15 min | Aide-mémoire & clôture | Ressources, questions |

> 💡 **Comment utiliser ce notebook** : chaque section alterne explications, démonstrations et exercices **🧪 À vous de jouer**. Faites les exercices avant de regarder la solution, cachée dans un bloc repliable juste en dessous.

> ⚠️ **Installations nécessaires** :
> ```bash
> pip install pandas numpy matplotlib seaborn networkx pyvis geopandas folium geopy mapclassify
> ```
> **geopandas** est la bibliothèque de référence pour les données spatiales en Python — l'équivalent direct de `sf` en R, ou de ce que ferait QGIS en mode scriptable. **folium** produit des cartes interactives (fondées sur la bibliothèque JavaScript Leaflet), l'équivalent cartographique de `pyvis` pour les réseaux. **geopy** sert au géocodage et au calcul de distances réelles.

> 📁 **Données** : placez `auc.csv` dans un dossier `data/` à côté de ce notebook. Le fond de carte des États américains (`us-states.json`) est téléchargé directement depuis GitHub dans la section 4 — gardez une connexion internet active à ce moment du cours.

> ✅ **Note** : l'ensemble du code de ce notebook (sections réseaux **et** cartographie) a été vérifié sur les données réelles de `auc.csv`. L'exercice à faire à la maison (`Exercice3_Python_reseaux.ipynb`), lui, continue d'utiliser `youmei.csv` et les jeux de données d'affiliation associés — deux terrains différents pour consolider les mêmes compétences.


# 0. Introduction : réseaux et cartes, deux grammaires du lien

Les Séances 1 et 2 nous ont appris à traiter des données comme des **tableaux** (lignes/colonnes) puis comme du **texte**. Aujourd'hui, nous ajoutons deux nouvelles façons de représenter une même réalité sociale :

- **L'analyse de réseau** répond à la question : *qui (ou quoi) est relié à qui (ou quoi) ?* Elle réorganise les données autour de **relations** (affiliations, collaborations, citations...) plutôt qu'autour d'observations indépendantes.
- **L'analyse spatiale** répond à la question : *qui (ou quoi) se trouve où ?* Elle réorganise les données autour de **positions géographiques**, et permet de révéler des structures — proximité, dispersion, concentration régionale — invisibles dans un tableau ou même dans un réseau.

Nous allons explorer les deux approches sur un **seul et même jeu de données** aujourd'hui : `auc.csv`, les parcours des étudiants de l'*American University Club of China* (Séance 1). Ce jeu de données a en réalité une double nature, que nous allons exploiter successivement :

- une nature **relationnelle** : chaque étudiant relie une université (où il/elle a étudié) à un employeur (où il/elle travaille en 1936) — la matière du réseau des sections 2 et 3 ;
- une nature **spatiale** : chaque université est située dans un État américain (`State`), et chaque étudiant s'est réinstallé dans une ville précise en 1936 (`City`) — la matière de la cartographie des sections 4 à 6.

En section 7, nous irons jusqu'à combiner les deux, en plaçant un réseau directement sur une carte : le même jeu de données révèle alors une véritable **carte de flux migratoires** entre universités américaines et villes chinoises.


# Qu'est-ce qu'un réseau ?

Un **réseau** (ou **graphe**) est une structure de données composée de :

- des **nœuds** (*nodes*, ou *vertices*) : les entités étudiées (personnes, institutions, lieux, mots...) ;
- des **liens** (*edges*, ou *arêtes*) : les relations entre ces entités (collaboration, affiliation, co-occurrence, correspondance...).

C'est une structure fondamentalement différente du tableau (dataframe) que nous avons manipulé lors des deux premières séances : plutôt que des lignes indépendantes, on s'intéresse ici aux **relations** entre unités. C'est un changement de perspective précieux pour les SHS : il permet d'étudier des phénomènes relationnels — réseaux de sociabilité, circulations, filiations institutionnelles, structures argumentatives — qui échappent à l'analyse tabulaire classique.

## Vocabulaire de base

| Terme | Description |
|---|---|
| **Nœud** (*node*/*vertex*) | Une entité du réseau (un individu, une institution, un lieu...) |
| **Lien** (*edge*) | Une relation entre deux nœuds |
| **Graphe non-orienté** | Les liens n'ont pas de direction (ex. « A et B ont co-écrit un article ») |
| **Graphe orienté** (*directed*) | Les liens ont un sens (ex. « A cite B ») |
| **Graphe pondéré** (*weighted*) | Chaque lien porte un poids (ex. nombre de collaborations) |
| **Réseau biparti** (*bipartite*) | Deux types de nœuds distincts, les liens n'existant qu'*entre* les deux types (ex. individus ↔ institutions), jamais entre nœuds d'un même type |
| **Projection** | Transformation d'un réseau biparti en réseau à un seul type de nœuds (ex. institutions liées entre elles via les individus qu'elles partagent) |
| **Degré** (*degree*) | Nombre de liens d'un nœud |
| **Centralité** | Famille de mesures quantifiant l'importance d'un nœud dans le réseau (voir section 3) |
| **Communauté** | Sous-groupe de nœuds densément connectés entre eux, plus faiblement au reste du réseau |

## Pourquoi l'analyse de réseau en SHS ?

L'analyse de réseau (*Social Network Analysis*, SNA) a une longue tradition en sciences sociales et en histoire — des travaux fondateurs de Padgett et Ansell sur les réseaux de mariage et d'affaires des Médicis, aux études contemporaines de circulation des idées, des correspondances savantes, ou des filières migratoires. 

## La bibliothèque NetworkX

**NetworkX** est la bibliothèque de référence pour la création, la manipulation et l'analyse de réseaux en Python — l'équivalent direct du package `igraph` (ou `tidygraph`/`ggraph`) en R.

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pyvis 
from networkx.algorithms import bipartite

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

print("Bibliothèques chargées avec succès ✅")


# 1. Prise en main de NetworkX

Avant de construire un réseau à partir de nos données réelles, familiarisons-nous avec les objets de base de NetworkX sur un petit exemple créé à la main.

## Créer et dessiner un graphe

Il existe quatre classes de graphes principales dans NetworkX : `nx.Graph()` (non orienté), `nx.DiGraph()` (orienté), et leurs variantes `MultiGraph`/`MultiDiGraph` autorisant les liens multiples. Pour l'essentiel de nos usages aujourd'hui, un simple `nx.Graph()` suffira.


In [ ]:
G = nx.Graph()

G.add_nodes_from(["Alice", "Bob", "Chen", "Deepa", "Emeka"])
G.add_edges_from([
    ("Alice", "Bob"),
    ("Alice", "Chen"),
    ("Bob", "Chen"),
    ("Chen", "Deepa"),
    ("Deepa", "Emeka"),
])

print("Nœuds :", G.nodes())
print("Liens :", G.edges())
print("Degré de chaque nœud :", dict(G.degree()))


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, ax=ax, with_labels=True, node_color="#4C72B0", font_color="white",
        node_size=800, edge_color="gray", width=1.5)
ax.set_title("Mon premier réseau")
plt.show()


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 <code>nx.spring_layout()</code> calcule une disposition des nœuds fondée sur un algorithme physique (répulsion entre nœuds, ressorts sur les liens) — la disposition la plus utilisée par défaut. <code>seed=42</code> garantit un résultat reproductible.
</div>

## Attributs et pondération

On peut attacher des attributs aux nœuds et des poids aux liens — deux notions que nous retrouverons systématiquement aujourd'hui :

In [ ]:
G.nodes["Alice"]["role"] = "Historian"
G.nodes["Bob"]["role"] = "Sociologist"
G.nodes["Chen"]["role"] = "Historian"
G.nodes["Deepa"]["role"] = "Linguist"
G.nodes["Emeka"]["role"] = "Sociologist"

nx.set_edge_attributes(G, 1, "weight")   # valeur par défaut, pour que chaque lien ait un poids
G["Alice"]["Bob"]["weight"] = 3          # 3 collaborations
G["Bob"]["Chen"]["weight"] = 5
G["Chen"]["Deepa"]["weight"] = 2

# On peut relire ces attributs
print(G.nodes(data=True))
print(G.edges(data=True))

On peut utiliser ces attributs pour enrichir la visualisation — par exemple, colorer les nœuds selon leur rôle et faire varier l'épaisseur des liens selon leur poids :

In [ ]:
role_colors = {"Historian": "#4C72B0", "Sociologist": "#DD8452", "Linguist": "#55A868"}
node_colors = [role_colors[G.nodes[n]["role"]] for n in G.nodes()]
edge_widths = [G[u][v]["weight"] for u, v in G.edges()]

fig, ax = plt.subplots(figsize=(6, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw(
    G, pos, ax=ax, with_labels=True,
    node_color=node_colors, font_color="white",
    node_size=800, edge_color="gray", width=edge_widths
)
ax.set_title("Réseau coloré par discipline, pondéré par le nombre de collaborations")
plt.show()

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Rappel : utiliser <code>sns.color_palette()</code> pour obtenir la palette. Pour afficher l'index de la couleur : 
<code>colors = sns.color_palette("deep", 3).as_hex()<code>
<code>print(colors)<code>

</div> 

## Le degré d'un nœud

Le **degré** d'un nœud est le nombre de liens qui lui sont rattachés — la mesure la plus simple, mais souvent très informative :

In [ ]:
dict(G.degree())

Cette mesure simple ne tient cependant pas compte du poids des liens. On peut l'intégrer facilement dans le calcul en utilisant le paramètre "weight": 

In [ ]:
dict(G.degree(weight="weight"))

## 🧪 À vous de jouer — Exercice 1

1. Créez un nouveau graphe non-orienté `G2` avec au moins 6 nœuds et 7 liens de votre choix (par exemple, un petit réseau de personnages d'un roman, ou de collègues d'un même laboratoire).
2. Attribuez à chaque nœud un attribut `group` (au moins deux groupes différents).
3. Calculez le degré de chaque nœud. Quel est le nœud le plus connecté ?
4. Dessinez ce graphe en colorant les nœuds selon leur groupe et en faisant varier la taille des nœuds selon leur degré.

In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir un exemple de solution</summary>

```python

G2 = nx.Graph()
G2.add_edges_from([
    ("A", "B"), ("A", "C"), ("B", "C"), ("C", "D"),
    ("D", "E"), ("D", "F"), ("E", "F")
])

groups = {"A": "group1", "B": "group1", "C": "group1", "D": "group2", "E": "group2", "F": "group2"}
nx.set_node_attributes(G2, groups, "group")

colors = {"group1": "#4C72B0", "group2": "#DD8452"}
node_colors = [colors[G2.nodes[n]["group"]] for n in G2.nodes()]

degrees = dict(G2.degree())
print(max(degrees.items(), key=lambda x: x[1])) # nœud de plus haut degré

fig, ax = plt.subplots(figsize=(6, 6))
pos2 = nx.spring_layout(G2, seed=1)
sizes = [degrees[n] * 300 for n in G2.nodes()]
nx.draw(G2, pos2, ax=ax, with_labels=True, node_color=node_colors, font_color="white", node_size=sizes)
plt.show()

```
</details>


# 2. Construire un réseau d'affiliation à partir de `auc.csv`

Nous retrouvons le jeu de données de la Séance 1 : les parcours d'étudiants formés aux États-Unis, avec leur université de formation et leur employeur après diplomation. Aujourd'hui, nous allons le traiter comme un véritable **réseau**, ce qui nous permettra d'aller plus loin : mesurer les liens entre individus et institutions, détecter des regroupements, etc.

## Recharger et nettoyer les données

On reprend les étapes de nettoyage vues en Séance 1 : 


In [ ]:
auc = pd.read_csv("data/auc.csv", sep=";", encoding="utf-8")

auc_simple = auc[["Name_full", "University", "State", "Field_main", "Employer_main", "Sector_1", "City"]].rename(columns={
    "Name_full": "Name",
    "Field_main": "Field",
    "Employer_main": "Employer",
    "Sector_1": "Sector"
})

auc_simple.head()


## Construire un réseau biparti : universités ↔ employeurs

Un **réseau biparti** (*bipartite network*) comporte deux types de nœuds distincts, les liens n'existant qu'*entre* les deux types. Ici : d'un côté les **universités**, de l'autre les **employeurs**, et un lien entre une université et un employeur chaque fois qu'un étudiant est passé de l'une à l'autre.

### Étape 1 : construire la table des liens (edge list)

On part des couples uniques (étudiant, université, employeur), puis on compte, pour chaque couple (université, employeur), le nombre d'étudiants qui l'ont emprunté :

In [ ]:
links = auc_simple[["Name", "University", "Employer"]].drop_duplicates().dropna(subset=["University", "Employer"])      # on sélectionne les paires uniques université-employeur sans valeurs manquantes

edges = links.groupby(["University", "Employer"]).size().reset_index(name="weight")      # on compte le nombre de paires que l'on transforme en "poids" 
edges = edges.sort_values("weight", ascending=False)                                     # on trie par ordre décroissant de "poids"   

print(f"{len(edges)} couples uniques université-employeur")
edges.head(10)


### Étape 2 : construire le graphe

In [ ]:
B = nx.Graph()          # on crée un graphe non orienté vide ('B' pour biparti, mais vous pouvez changer)

universities = edges["University"].unique()         # on récupère la liste des universités
employers = edges["Employer"].unique()              # on récupère la liste des employeurs

B.add_nodes_from(universities, bipartite=0, node_type="University")     # on ajoute les universités au graphe B et on leur attribue un type dans le réseau bipartite (0 est une simple convention)
B.add_nodes_from(employers, bipartite=1, node_type="Employer")          # on ajoute les employers au graphe B et on leur attribue un autre type dans le réseau bipartite (1 est une convention)

for _, row in edges.iterrows():
    B.add_edge(row["University"], row["Employer"], weight=row["weight"])    # on crée les liens du graphe et on leur attribue le poids correspondant à chaque paire

print(f"{B.number_of_nodes()} nœuds ({len(universities)} universités + {len(employers)} employeurs)")
print(f"{B.number_of_edges()} liens")
print(f"Densité : {nx.density(B):.4f}")
print(f"Composantes connexes : {nx.number_connected_components(B)}")


### Étape 3 : visualiser le réseau biparti

Le réseau complet est trop dense pour être lisible d'un seul coup d'œil. On commence donc par filtrer les liens les plus significatifs (poids ≥ 2, c'est-à-dire au moins deux étudiants ayant emprunté ce chemin) :

In [ ]:
strong_edges = edges[edges["weight"] >= 2]

B_strong = nx.Graph()
B_strong.add_nodes_from(strong_edges["University"].unique(), bipartite=0, node_type="University")
B_strong.add_nodes_from(strong_edges["Employer"].unique(), bipartite=1, node_type="Employer")
for _, row in strong_edges.iterrows():
    B_strong.add_edge(row["University"], row["Employer"], weight=row["weight"])

fig, ax = plt.subplots(figsize=(11, 9))
pos = nx.spring_layout(B_strong, seed=42, k=0.5)
node_colors = ["#4C72B0" if B_strong.nodes[n]["node_type"] == "University" else "#DD8452" for n in B_strong.nodes()]
edge_widths = [B_strong[u][v]["weight"] * 0.8 for u, v in B_strong.edges()]

nx.draw_networkx_nodes(B_strong, pos, node_color=node_colors, node_size=250, ax=ax)
nx.draw_networkx_edges(B_strong, pos, width=edge_widths, edge_color="lightgray", ax=ax)
nx.draw_networkx_labels(B_strong, pos, font_size=6, ax=ax)
ax.set_title("Réseau biparti : universités (bleu) et employeurs (orange)")
ax.axis("off")

plt.figtext(
    0.5, 0.01,
    "Note : seuls les liens de poids ≥ 2 sont représentés.",
    ha="center",
    fontsize=9
)

plt.show()


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 <b>Le filtrage est une étape normale et souvent nécessaire</b> en visualisation de réseau : un réseau complet issu de données réelles est presque toujours trop dense pour être lu visuellement. On filtre en général selon un seuil de poids, ou en ne conservant que les *n* nœuds les plus connectés — à condition de toujours mentionner ce filtrage dans la légende ou le commentaire, pour ne pas donner une image trompeuse du réseau complet.
</div>


## Mesures de réseau et projection

Au-delà de la seule visualisation, l'analyse de réseau offre des **mesures quantitatives** pour caractériser la structure d'un graphe et l'importance relative de ses nœuds.

### Densité

La **densité** d'un réseau est la proportion de liens existants par rapport à tous les liens possibles (entre 0 = aucun lien, et 1 = réseau complet où tous les nœuds sont connectés entre eux) :

In [ ]:
nx.density(B)

## Composantes connexes

Une **composante connexe** est un sous-ensemble de nœuds tous reliés entre eux (directement ou indirectement), mais déconnectés du reste du réseau. Un réseau réel comporte souvent plusieurs composantes (des "îlots" isolés) :

In [ ]:
print("Nombre de composantes connexes :", nx.number_connected_components(B))

# Taille de chaque composante
sizes = [len(c) for c in nx.connected_components(B)]
sorted(sizes, reverse=True)[:10]

In [ ]:
# On isole la plus grande composante, souvent la plus intéressante à analyser
largest_cc = max(nx.connected_components(B), key=len)
B_main = B.subgraph(largest_cc).copy()

print(f"Composante principale : {B_main.number_of_nodes()} nœuds, {B_main.number_of_edges()} liens")

## Projection : : passer d'un réseau biparti à un réseau à un seul type de nœuds

Le réseau biparti université ↔ employeur est utile, mais il ne permet pas directement de répondre à des questions comme « quelles universités partagent le plus de destinées professionnelles communes ? ». Pour cela, on **projette** le réseau biparti sur l'un des deux côtés — ici, le côté « université » : deux universités sont reliées si elles partagent au moins un employeur, le poids du lien reflétant l'intensité de ce partage.

In [ ]:
univ_nodes = {n for n, d in B.nodes(data=True) if d["node_type"] == "University"}

G_univ = bipartite.weighted_projected_graph(B, univ_nodes)

print(f"{G_univ.number_of_nodes()} universités, {G_univ.number_of_edges()} liens (employeurs partagés)")


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 <code>bipartite.weighted_projected_graph(B, univ_nodes)</code> calcule automatiquement, pour chaque paire d'universités, le nombre d'employeurs qu'elles ont en commun, et l'utilise comme poids du nouveau lien. C'est l'équivalent réseau de la « déduplication + comptage » que l'on ferait à la main avec un `.groupby()`.
</div>


## 🧪 À vous de jouer — Exercice 2

1. Combien y a-t-il, au total, de couples (université, employeur) uniques dans `edges` (avant filtrage) ?
2. Quels sont les 5 employeurs ayant recruté des diplômés du plus grand nombre d'universités différentes ? (indice : c'est une question sur le **degré** des nœuds employeurs dans `B`)
3. Reconstruisez le réseau biparti en ne conservant que les étudiants du champ `Field == "Engineering"`. Combien de nœuds et de liens ce sous-réseau contient-il ?
4. Redessinez le réseau filtré (`B_strong`) en utilisant un seuil de poids différent (par exemple ≥ 3) : le réseau est-il plus ou moins lisible ?
5. Combien de composantes connexes compte `G_univ` ? Que nous apprend ce nombre sur la structure du réseau ?
6. Quelles sont les 3 universités reliées par le plus grand nombre d'employeurs partagés (regardez le poids des liens de `G_univ`) ?

In [ ]:
# 🧪 Essayez ici

<details>
<summary>▶️ Voir la solution</summary>

```python
# Question 1
len(edges)

# Question 2
degrees = dict(B.degree())
employer_degrees = {n: d for n, d in degrees.items() if B.nodes[n]["node_type"] == "Employer"}
sorted(employer_degrees.items(), key=lambda x: -x[1])[:5]

# Question 3
links_eng = auc_simple[auc_simple["Field"] == "Engineering"][["Name", "University", "Employer"]].drop_duplicates().dropna()
edges_eng = links_eng.groupby(["University", "Employer"]).size().reset_index(name="weight")

B_eng = nx.Graph()
B_eng.add_nodes_from(edges_eng["University"].unique(), bipartite=0, node_type="University")
B_eng.add_nodes_from(edges_eng["Employer"].unique(), bipartite=1, node_type="Employer")
for _, row in edges_eng.iterrows():
    B_eng.add_edge(row["University"], row["Employer"], weight=row["weight"])

print(B_eng.number_of_nodes(), B_eng.number_of_edges())

# Question 4
strong_edges_3 = edges[edges["weight"] >= 3]
# ... reconstruire le graphe et le dessiner comme ci-dessus, en remplaçant le seuil

# Question 5
nx.number_connected_components(G_univ)
# 13 composantes : le réseau est loin d'être un seul bloc homogène — plusieurs groupes
# d'universités ne partagent jamais le même employeur entre eux.

# Question 6
sorted(G_univ.edges(data=True), key=lambda x: -x[2]["weight"])[:3]
```
</details>


# 3. Centralités et communautés

## Centralité : qui est le plus important dans le réseau ?

La **centralité** est une famille de mesures visant à quantifier l'importance d'un nœud. Il existe différentes définitions de ce qu'« être important » signifie dans un réseau :

| Mesure | Ce qu'elle capture | Question posée |
|---|---|---|
| **Degré** (*degree centrality*) | Nombre de connexions directes | « Combien de partenaires ce nœud a-t-il ? » |
| **Intermédiarité** (*betweenness centrality*) | Fréquence à laquelle un nœud se trouve sur le plus court chemin entre deux autres | « Ce nœud sert-il de pont/passage obligé entre d'autres parties du réseau ? » |
| **Proximité** (*closeness centrality*) | Distance moyenne aux autres nœuds | « Ce nœud peut-il atteindre rapidement tout le reste du réseau ? » |
| **Vecteur propre** (*eigenvector centrality*) | Importance pondérée par l'importance de ses voisins | « Ce nœud est-il connecté à d'autres nœuds eux-mêmes importants ? » |

Calculons ces quatre mesures sur le réseau des universités :


In [ ]:
centrality = pd.DataFrame({
    "degree": nx.degree_centrality(G_univ),
    "betweenness": nx.betweenness_centrality(G_univ, weight="weight"), # si on veut tenir compte du poids des liens
    "closeness": nx.closeness_centrality(G_univ),
    "eigenvector": nx.eigenvector_centrality(G_univ)
})

centrality.sort_values("degree", ascending=False).head(10)


Columbia, Harvard et Yale dominent largement les classements — trois des institutions historiquement les plus fréquentées par les étudiants chinois de cette génération — ce qui n'est pas une surprise pour qui connaît un peu l'histoire de ces échanges universitaires.

<div class="alert alert-danger" role="alert" style="background-color:#f8d7da;padding:10px;border-radius:5px;">
⚠️ <b>Attention à l'interprétation</b> : un nœud avec un fort <b>degré</b> mais une faible <b>intermédiarité</b> est bien connecté localement, sans forcément jouer de rôle de « pont » entre différentes parties du réseau. Inversement, un nœud à faible degré mais forte intermédiarité peut être un intermédiaire stratégique, même peu visible à première vue. Croiser plusieurs mesures de centralité, plutôt que se fier à une seule, donne une image plus riche et plus fiable de la structure du réseau.
</div>

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Le réseau comporte plusieurs composantes connexes (voir exercice 2). Certaines mesures de centralité (notamment intermédiarité et vecteur propre) nécessitent un réseau connexe pour être bien définie. Il est donc préférable de les calculer seulement sur la plus grande composante.  
</div>
<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">  
💡 Noter aussi que la centralité de vecteur propre se calcule de manière itérative : l'algorithme part d'une estimation initiale, recalcule les scores, puis recommence jusqu'à ce que les valeurs se stabilisent suffisamment. Par défaut, NetworkX utilise 100 itérations. Si le réseau est complexe et que l'algorithme ne converge pas en 100 itérations, il est possible d'augmenter avec l'argument <code>max_iter = ...</code>.
</div>

In [ ]:
largest_cc = max(nx.connected_components(G_univ), key=len)
G_univ_main = G_univ.subgraph(largest_cc).copy()

eigenvector = pd.Series(nx.eigenvector_centrality(G_univ_main, weight="weight", max_iter=1000), name="eigenvector")
eigenvector.sort_values(ascending=False).head(10)


On peut aussi visualiser le réseau des universités en faisant varier la **taille des nœuds** selon leur centralité de degré, pour repérer visuellement les institutions les plus centrales :

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))

pos = nx.spring_layout(G_univ, seed=42, k=0.6)
node_sizes = [centrality.loc[n, "degree"] * 4000 + 100 for n in G_univ.nodes()]
edge_widths = [G_univ[u][v]["weight"] * 0.5 for u, v in G_univ.edges()]

nx.draw_networkx_nodes(G_univ, pos, node_size=node_sizes, node_color="#4C72B0", alpha=0.85, ax=ax)
nx.draw_networkx_edges(G_univ, pos, width=edge_widths, edge_color="lightgray", ax=ax)
nx.draw_networkx_labels(G_univ, pos, font_size=7, ax=ax)

ax.set_title("University network (node size = degree centrality)")
ax.axis("off")
plt.show()


## Détection de communautés avec l'algorithme de Louvain

### Qu'est-ce qu'une communauté ?

Une **communauté** (ou *cluster*) est un sous-ensemble de nœuds **densément connectés entre eux**, mais plus faiblement reliés au reste du réseau. Détecter des communautés permet de faire émerger des regroupements qui ne sont pas immédiatement visibles dans les données d'origine.

### L'algorithme de Louvain

L'algorithme de **Louvain** est la méthode de référence pour la détection de communautés : il cherche à maximiser la **modularité** du réseau — une mesure de la qualité d'un découpage en communautés (des liens denses à l'intérieur des groupes, rares entre eux). NetworkX l'intègre nativement : 

In [ ]:
communities_univ = nx.community.louvain_communities(G_univ, weight="weight", seed=42)   

print(f"{len(communities_univ)} communautés d'universités détectées")
print(f"Modularité : {nx.community.modularity(G_univ, communities_univ, weight='weight'):.3f}")

for i, com in enumerate(communities_univ):
    if len(com) > 1:
        print(f"Communauté {i} ({len(com)} universités) :", sorted(com)[:8])


In [ ]:
# Pour trier les communautés selon leur taille (décroissante) :

communities_sorted = sorted(
    enumerate(communities_univ),
    key=lambda x: len(x[1]),
    reverse=True
)

for i, com in communities_sorted:
    if len(com) > 1:
        print(
            f"Communauté {i} ({len(com)} universités) :",
            sorted(com)[:8]
        )

On peut mesurer la **modularité** du découpage obtenu (plus elle est élevée, plus la structure en communautés est marquée) :

In [ ]:
nx.community.modularity(G_univ, communities_univ, weight="weight")


<div class="alert alert-danger" role="alert" style="background-color:#f8d7da;padding:10px;border-radius:5px;">
⚠️ <b>Attention à l'interprétation</b> : Une modularité élevée ne signifie pas nécessairement que la partition obtenue est la plus pertinente d'un point de vue sociologique. La modularité est un indicateur statistique qui mesure la structuration du réseau en groupes relativement denses ; elle ne renseigne pas, à elle seule, sur la signification sociale de ces groupes. Il est donc essentiel d'examiner la composition des communautés obtenues et de les interpréter au regard de la question de recherche et de la connaissance du terrain.
</div>
<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 D'autres algorithmes de détection de communautés sont disponibles dans NetworkX (voir la <a href="https://networkx.org/documentation/latest/reference/algorithms/community.html">documentation</a>). Comparer les partitions obtenues avec différentes méthodes permet d'éviter de s'enfermer dans une solution unique et d'affiner l'interprétation des communautés identifiées.
</div>

### Visualiser les communautés

Pour visualiser ce découpage, on attribue à chaque nœud la couleur de sa communauté : 

In [ ]:
# On crée un dictionnaire nœud -> numéro de communauté

node_to_community = {}
for i, com in enumerate(communities_univ):
    for node in com:
        node_to_community[node] = i
nx.set_node_attributes(G_univ, node_to_community, "community")

palette = sns.color_palette("tab20", len(communities_univ)).as_hex()

fig, ax = plt.subplots(figsize=(11, 9))
pos = nx.spring_layout(G_univ, seed=42, k=0.6)
node_colors = [palette[G_univ.nodes[n]["community"] % len(palette)] for n in G_univ.nodes()]
node_sizes = [centrality.loc[n, "degree"] * 4000 + 100 for n in G_univ.nodes()]         # On ajuste la taille des nœuds selon leur centralité de degré.

nx.draw_networkx_nodes(G_univ, pos, node_size=node_sizes, node_color=node_colors, alpha=0.9, ax=ax)
nx.draw_networkx_edges(G_univ, pos, width=1, edge_color="lightgray", ax=ax)
nx.draw_networkx_labels(G_univ, pos, font_size=6, ax=ax)
ax.set_title("Réseau des universités — couleur = communauté, taille = centralité de degré")
ax.axis("off")
plt.show()


### Visualisation interactive avec pyvis

**pyvis** permet de produire des réseaux **interactifs** (zoom, déplacement des nœuds à la souris, info-bulles) exportés en HTML — bien plus confortables à explorer qu'une image statique pour les réseaux de grande taille.

In [ ]:
from pyvis.network import Network

degree = dict(G_univ.degree())
weighted_degree = dict(G_univ.degree(weight="weight"))

net2 = Network(
    height="750px",
    width="100%",
    notebook=True,
    cdn_resources="in_line",
    bgcolor="white",
    select_menu=True,
    filter_menu=True
)

for node, data in G_univ.nodes(data=True):

    net2.add_node(
        node,
        label=node,

        group=data["community"],
        community=data["community"],

        degree=degree[node],
        weighted_degree=weighted_degree[node],

        value=weighted_degree[node],

        title=(
            f"<b>{node}</b><br>"
            f"Communauté : {data['community']}<br>"
            f"Degré : {degree[node]}<br>"
            f"Degré pondéré : {weighted_degree[node]:.1f}"
        )
    )

for u, v, data in G_univ.edges(data=True):

    net2.add_edge(
        u,
        v,
        value=data["weight"],
        weight=data["weight"],
        title=f"Poids : {data['weight']}"
    )

net2.show_buttons(filter_=["physics"])      # pour afficher le panneau de contrôle
# net2.toggle_physics(False)                # pour stabiliser le mouvement 

net2.show("university_network_communities.html")

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Le fichier <code>university_network_communities.html</code> peut être ouvert dans n'importe quel navigateur et partagé indépendamment de Python — un bon format pour intégrer une visualisation de réseau interactive dans un carnet de recherche ou un site web.
</div>

## 🧪 À vous de jouer — Exercice 3

1. Quelle université a la plus forte intermédiarité (*betweenness*) ? Est-ce la même que celle qui a le plus fort degré ?
2. Filtrez `G_univ` pour ne garder que les liens de poids ≥ 2 avant de calculer les centralités : les classements changent-ils ? 
3. Essayez un autre algorithme de détection de communautés disponible dans NetworkX, par exemple `nx.community.greedy_modularity_communities(G_univ, weight="weight")`. Obtenez-vous un découpage similaire à celui de Louvain ?
4. **Bonus**: Réalisez la même projection, mais du côté des **employeurs** cette fois (`bipartite.weighted_projected_graph(B, employer_nodes)`). Combien de composantes connexes compte cette projection ? Quelle interprétation en tirer ? 
5. Quel employeur possède la plus forte centralité de vecteur propre ? D'intermédiarité ? 
6. Combien de communautés peuvent être trouvées dans ce réseau selon l'algorithme de Louvain ? 


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir la solution</summary>

```python
# Question 1
centrality.sort_values("betweenness", ascending=False).head(5)

# Question 2
G_univ_strong = nx.Graph(((u, v, d) for u, v, d in G_univ.edges(data=True) if d["weight"] >= 2))
pd.Series(nx.degree_centrality(G_univ_strong)).sort_values(ascending=False).head(10)

# Question 3
communities_greedy = nx.community.greedy_modularity_communities(G_univ, weight="weight")
len(communities_greedy)

# Question 4 (bonus)
employer_nodes = {n for n, d in B.nodes(data=True) if d["node_type"] == "Employer"}
G_empl = bipartite.weighted_projected_graph(B, employer_nodes)

nx.number_connected_components(G_empl)

# Question 5

empl_eigenvector = pd.Series(nx.eigenvector_centrality(G_empl)).sort_values(ascending=False)
empl_eigenvector.head(10)

empl_eigenvector = pd.Series(nx.eigenvector_centrality(G_empl)).sort_values(ascending=False)
empl_eigenvector.head(10)

# Question 6

empl_communities = nx.community.louvain_communities(G_empl, weight="weight", seed=42) 
len(empl_communities)

for i, com in enumerate(empl_communities):
    if len(com) > 1:
        print(f"Communauté {i} ({len(com)} employeurs) :", sorted(com)[:8])

```
</details>

---

📌 **Pour aller plus loin sur les réseaux** : la fin de ce notebook (en annexe) applique exactement la même logique (réseau, projection, centralité, communautés) à un réseau construit à partir d'un corpus de texte plutôt que d'un tableau — un réseau de **co-occurrence d'entités nommées**, à partir du corpus de presse olympique de la Séance 2. Nous n'aurons pas le temps de le voir en séance, mais n'hésitez pas à vous y référer chez vous.


# 4. Introduction à geopandas

Changeons de registre, sans changer de jeu de données. **geopandas** étend pandas pour manipuler des données **spatiales** : chaque ligne d'un `GeoDataFrame` est une observation ordinaire, mais possède en plus une colonne spéciale, `geometry`, qui décrit sa forme dans l'espace (un point, une ligne, ou un polygone).

## Trois notions clés avant de commencer

| Notion | Description |
|---|---|
| **Géométrie** (*geometry*) | La forme spatiale associée à une observation : `Point` (un lieu ponctuel), `LineString` (une ligne, ex. une route), `Polygon` (une surface, ex. une frontière administrative). |
| **Système de coordonnées** (*CRS*, *Coordinate Reference System*) | La façon dont les coordonnées (des nombres) sont rattachées à des positions réelles sur Terre. Le CRS le plus courant, **EPSG:4326** (WGS 84), exprime les positions en degrés de latitude/longitude — c'est celui du GPS. |
| **Projection** | Une transformation mathématique du CRS géographique (degrés, sur une sphère) vers un CRS **projeté** (mètres, sur un plan) — nécessaire pour calculer des distances ou des surfaces de façon fiable. |

<div class="alert alert-danger" role="alert" style="background-color:#f8d7da;padding:10px;border-radius:5px;">
⚠️ <b>À garder en tête toute la journée</b> : un même jeu de coordonnées peut correspondre à des lieux différents selon le CRS dans lequel on l'interprète. Vérifiez <b>toujours</b> le CRS d'un fichier spatial avant de le combiner avec un autre (<code>gdf.crs</code>), et reprojetez si nécessaire avec <code>.to_crs()</code>.
</div>

## Charger un fond de carte

Nous allons cartographier la variable `State` de `auc.csv` — l'État américain où chaque université est localisée. Il nous faut donc un fond de carte des États-Unis, au format GeoJSON :

In [ ]:
!pip install geopandas
import geopandas as gpd

us_states = gpd.read_file("https://raw.githubusercontent.com/python-visualization/folium/main/examples/data/us-states.json")

us_states.head()


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 <code>gpd.read_file()</code> est l'équivalent de <code>pd.read_csv()</code>, mais pour des formats spatiaux (GeoJSON, Shapefile, GeoPackage...). Il peut lire directement une URL, exactement comme pandas.
</div>

Un `GeoDataFrame` s'inspecte comme un DataFrame ordinaire :

In [ ]:
print(type(us_states))                  # type de données 
print(us_states.shape)                  # nombre de lignes et de colonnes 
print(us_states.columns.tolist())       # nom des colonnes
print(us_states.crs)                    # le système de coordonnées du fond de carte


**`us_states.crs`** devrait afficher `EPSG:4326` : les coordonnées sont exprimées en degrés de latitude/longitude — le standard pour la plupart des données géographiques mondiales en libre accès.

## Un premier tracé

La méthode `.plot()` d'un GeoDataFrame trace directement les géométries :

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
us_states.plot(ax=ax, color="#EFEAD9", edgecolor="#8a8060", linewidth=0.5)
ax.set_title("Les 50 États des États-Unis")
ax.axis("off")
plt.show()


In [ ]:
# Zoom sur les États-Unis continentaux 

fig, ax = plt.subplots(figsize=(10, 6))
us_states.plot(ax=ax, color="#EFEAD9", edgecolor="#8a8060", linewidth=0.5)

ax.set_xlim(-125, -66)
ax.set_ylim(24, 50) 

# Ajouter le nom des États
for _, row in us_data.iterrows():
    point = row.geometry.representative_point()

    ax.annotate(
        text=row["name"],
        xy=(point.x, point.y),
        ha="center",
        va="center",
        fontsize=7
    )
    
ax.axis("off")
ax.set_title("Les 48 États continentaux")
plt.show()


## 🧪 À vous de jouer — Exercice 4

1. Combien d'États (lignes) contient `us_states` ?
2. Affichez uniquement la géométrie de l'État de New York (indice : filtrez comme un DataFrame classique, sur la colonne `name`).
3. Tracez uniquement cet État sur une carte.


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir la solution</summary>

```python
# Question 1
us_states.shape[0]

# Question 2
new_york = us_states[us_states["name"] == "New York"]
new_york

# Question 3
fig, ax = plt.subplots(figsize=(5, 5))
new_york.plot(ax=ax, color="#4C72B0")
ax.set_title("New York")
ax.axis("off")
plt.show()
```
</details>


# 5. Cartographie choroplèthe : dans quels États les étudiants se sont-ils formés ?

Une **carte choroplèthe** colore chaque unité géographique (ici, un État) selon la valeur d'une variable (ici, le nombre d'étudiants formés dans cet État). C'est l'équivalent cartographique d'un barplot : au lieu de comparer des catégories sur un axe, on les compare dans l'espace.

## Préparer les données

On reprend `auc`, et la variable `State` :

In [ ]:
state_counts = auc["State"].value_counts().reset_index()
state_counts.columns = ["state", "n"]

state_counts.head(15)


<div class="alert alert-danger" role="alert" style="background-color:#f8d7da;padding:10px;border-radius:5px;">
⚠️ <b>Nettoyage nécessaire avant de cartographier</b> : la colonne <code>State</code> ne contient pas <i>uniquement</i> des États américains. Un petit nombre d'étudiants ont été formés hors des États-Unis (<code>Country_edu</code> le confirme), et la colonne <code>State</code> indique alors un pays ou une ville étrangère (<code>Shanghai</code>, <code>Philippines</code>, <code>Scotland</code>, <code>Tokyo</code>...). De même, <code>District of Columbia</code> est un district fédéral, pas un État — il n'apparaîtra donc naturellement pas sur un fond de carte des 50 États.
</div>


In [ ]:
not_a_state = set(state_counts["state"]) - set(us_states["name"])
print("Valeurs non cartographiables sur ce fond de carte (hors des 50 États) :")
print(not_a_state)

n_excluded = state_counts[state_counts["state"].isin(not_a_state)]["n"].sum()
print(f"\n{n_excluded} étudiants concernés, sur {state_counts['n'].sum()} au total — à mentionner explicitement dans toute carte ou légende produite à partir de ces données.")


## Fusionner les données avec la géométrie

On fusionne (`.merge()`, comme en Séance 1) le fond de carte avec nos comptages, sur le nom de l'État :

In [ ]:
us_data = us_states.merge(state_counts, left_on="name", right_on="state", how="left")
us_data["n"] = us_data["n"].fillna(0)   # États sans étudiant enregistré

us_data[us_data["n"] > 0][["name", "n"]].sort_values("n", ascending=False)


## Carte choroplèthe statique

On trace la carte en coloriant chaque État selon `n`, avec l'argument `column=` de `.plot()` :

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))

us_data.plot(
    column="n",
    ax=ax,
    cmap="YlOrRd",
    edgecolor="#8a8060",
    linewidth=0.4,
    legend=True,
    missing_kwds={
        "color": "#EFEAD9",
        "label": "Aucun étudiant"
    }
)

# Zoom sur les États-Unis continentaux 
ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)

ax.set_title(
    "Distribution des étudiants par État américain (annuaire de 1936)",
    fontsize=20,
    fontweight="bold",
    pad=20
)

# Ajouter le nom des États
for _, row in us_data.iterrows():
    point = row.geometry.representative_point()

    ax.annotate(
        text=row["name"],
        xy=(point.x, point.y),
        ha="center",
        va="center",
        fontsize=7
    )

ax.axis("off")

plt.tight_layout()
plt.show()

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 <code>missing_kwds</code> définit l'apparence des États sans donnée (<code>n == 0</code> ou <code>NaN</code>) — toujours les distinguer visuellement des États à faible valeur, pour ne pas laisser croire à une absence réelle d'étudiants plutôt qu'à une absence de données.
</div>

Sans surprise pour qui connaît la géographie universitaire américaine du début du XXe siècle, l'État de **New York** domine très largement (Columbia, NYU, Cornell y ont, entre autres, des campus ou antennes), suivi du **Massachusetts** (Harvard, MIT) et de l'**Illinois** (Chicago).

## Carte choroplèthe interactive avec folium

**folium** produit des cartes interactives (zoom, info-bulles) dans le navigateur, fondées sur la bibliothèque JavaScript Leaflet — l'équivalent cartographique de ce que `pyvis` fait pour les réseaux.

In [ ]:
!pip install folium
import folium



In [ ]:
m = folium.Map(location=[39, -98], zoom_start=4, tiles="OpenStreetMap")

folium.Choropleth(
    geo_data=us_data.__geo_interface__,
    data=state_counts,
    columns=["state", "n"],
    key_on="feature.properties.name",
    fill_color="YlOrRd",
    fill_opacity=0.8,
    line_opacity=0.3,
    legend_name="Nombre d'étudiants formés"
).add_to(m)

folium.GeoJson(
    us_data,
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "n"],
        aliases=["État :", "Nombre d'étudiants :"],
        localize=True,
        sticky=False
    ),
    style_function=lambda feature: {
        "fillOpacity": 0,
        "color": "transparent",
        "weight": 0
    }
).add_to(m)

m

# m.save("us_students_map.html")

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
<b>Explications</b>

- <code>location=[39, -98]</code> centre la carte sur les États-Unis (latitude, longitude), <code>zoom_start=4</code> définit le niveau de zoom initial.
- <code>geo_data=</code> attend un GeoJSON — <code>.__geo_interface__</code> convertit à la volée notre GeoDataFrame dans ce format.
- <code>key_on="feature.properties.name"</code> indique la clé de jointure côté géométrie (le nom de l'État dans le GeoJSON), à faire correspondre avec la colonne <code>"state"</code> côté données.
</div>

## 🧪 À vous de jouer — Exercice 5

1. Recalculez `state_counts` en pourcentage plutôt qu'en effectif brut, et refaites la carte choroplèthe statique avec cette nouvelle variable. Modifiez la palette de couleurs de la carte statique (essayez `cmap="Blues"` ou `cmap="viridis"`).
2. Cartographiez uniquement les étudiants de nationalité chinoise par État. La répartition diffère-t-elle de la carte incluant tout le monde ?
3. **Bonus**: construisez une version interactive de cette carte folium. Veillez à afficher les états et le nombre d'étudiants par Etats


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir la solution</summary>

```python
# Question 1
state_counts["pct"] = state_counts["n"] / state_counts["n"].sum() * 100
us_pct = us_states.merge(state_counts, left_on="name", right_on="state", how="left")
us_pct["pct"] = us_pct["pct"].fillna(0)

fig, ax = plt.subplots(figsize=(12, 7))
us_pct.plot(column="pct", ax=ax, cmap="Blues", legend=True, edgecolor="#8a8060", linewidth=0.4,
            missing_kwds={"color": "#EFEAD9"})
ax.set_title(
    "Pourcentage d'étudiants par État américain (annuaire de 1936)",
    fontsize=16,
    fontweight="bold",
    pad=20
)
ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)
ax.axis("off")
plt.show()

# Question 2

chinese_counts = auc[auc["Nationality"] == "Chinese"]["State"].value_counts().reset_index()
chinese_counts.columns = ["state", "n"]
us_chinese = us_states.merge(chinese_counts, left_on="name", right_on="state", how="left")
us_chinese["n"] = us_chinese["n"].fillna(0)

fig, ax = plt.subplots(figsize=(12, 7))
us_chinese.plot(column="n", ax=ax, cmap="PuRd", legend=True, edgecolor="#8a8060", linewidth=0.4,
                missing_kwds={"color": "#EFEAD9"})
ax.set_title("Distribution des étudiants chinois par État américain")
ax.set_xlim(-125, -66)
ax.set_ylim(24, 50)
ax.axis("off")
plt.show()

# Question 3 (bonus)

m_chinese = folium.Map(
    location=[39, -98],
    zoom_start=4,
    tiles="OpenStreetMap"
)

# 5. Ajouter la choroplèthe
folium.Choropleth(
    geo_data=us_chinese.__geo_interface__,
    data=us_chinese,
    columns=["name", "n"],
    key_on="feature.properties.name",
    fill_color="PuRd",
    fill_opacity=0.8,
    line_opacity=0.4,
    line_color="#8a8060",
    legend_name="Nombre d'étudiants chinois"
).add_to(m_chinese)

# 6. Ajouter les informations au survol
folium.GeoJson(
    us_chinese,
    style_function=lambda feature: {
        "fillOpacity": 0,
        "color": "transparent",
        "weight": 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "n"],
        aliases=["État :", "Nombre d'étudiants chinois :"],
        localize=True,
        sticky=False,
        labels=True
    )
).add_to(m_chinese)

# 7. Ajuster la vue aux États-Unis continentaux
m_chinese.fit_bounds([
    [24, -125],
    [50, -66]
])

m_chinese

# m_chinese.save("chinese_students_map.html")

```
</details>


In [ ]:
m_chinese = folium.Map(
    location=[39, -98],
    zoom_start=4,
    tiles="OpenStreetMap"
)

# 5. Ajouter la choroplèthe
folium.Choropleth(
    geo_data=us_chinese.__geo_interface__,
    data=us_chinese,
    columns=["name", "n"],
    key_on="feature.properties.name",
    fill_color="PuRd",
    fill_opacity=0.8,
    line_opacity=0.4,
    line_color="#8a8060",
    legend_name="Nombre d'étudiants chinois"
).add_to(m_chinese)

# 6. Ajouter les informations au survol
folium.GeoJson(
    us_chinese,
    style_function=lambda feature: {
        "fillOpacity": 0,
        "color": "transparent",
        "weight": 0
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["name", "n"],
        aliases=["État :", "Nombre d'étudiants chinois :"],
        localize=True,
        sticky=False,
        labels=True
    )
).add_to(m_chinese)

# 7. Ajuster la vue aux États-Unis continentaux
m_chinese.fit_bounds([
    [24, -125],
    [50, -66]
])

m_chinese

m_chinese.save("chinese_students_map.html")

# 6. Cartes de points, distances et cartes interactives

La section précédente cartographiait des **polygones** (les États). Regardons maintenant la colonne `City` : la ville où chaque étudiant s'est réinstallé en 1936, après ses études. C'est une donnée **ponctuelle** — chaque ville est un point précis, pas une région.


In [ ]:
auc["City"].value_counts(dropna=False)


Sans surprise, **Shanghai** concentre l'écrasante majorité des étudiants (c'est là qu'est publié l'annuaire de l'*American University Club of China*), mais une douzaine d'autres villes apparaissent — pour l'essentiel en Chine, avec quelques exceptions notables (New York, Pasadena, Singapour) correspondant aux étudiants qui ne sont pas rentrés.

## Localiser les villes

Comme pour les universités américaines, on part d'une petite table de coordonnées, préparée pour les villes présentes dans `auc.csv` (certaines sous leur ancienne romanisation historique — Peiping pour Pékin, Canton pour Guangzhou, Hankow pour Wuhan...) :

In [ ]:
city_coords = {
    "Shanghai": (31.2304, 121.4737),
    "Nanking": (32.0603, 118.7969),
    "Peiping": (39.9042, 116.4074),
    "Hangchow": (30.2741, 120.1551),
    "New York": (40.7128, -74.0060),
    "Soochow": (31.2989, 120.5853),
    "Singapore": (1.3521, 103.8198),
    "Pasadena, CA": (34.1478, -118.1445),
    "Haichow": (34.6039, 119.1248),
    "Tangshan (Hopei)": (39.6243, 118.1944),
    "Canton": (23.1291, 113.2644),
    "Hankow": (30.5928, 114.2999),
}

city_counts = auc["City"].value_counts().reset_index()
city_counts.columns = ["city", "n"]
city_counts["lat"] = city_counts["city"].map(lambda c: city_coords.get(c, (None, None))[0])
city_counts["lon"] = city_counts["city"].map(lambda c: city_coords.get(c, (None, None))[1])

city_counts


## Carte de points interactive

On place un marqueur pour chaque ville, avec une taille proportionnelle au nombre d'étudiants :

In [ ]:
m_cities = folium.Map(location=[30, 110], zoom_start=3, tiles="OpenStreetMap")

for _, row in city_counts.dropna(subset=["lat", "lon"]).iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=row["n"] ** 0.5 * 3,         # racine carrée : évite que Shanghai n'écrase tout le reste visuellement
        popup=f"{row['city']} ({row['n']} étudiants)",
        color="#C44E52", fill=True, fill_opacity=0.7
    ).add_to(m_cities)

m_cities

# m_cities.save("m_cities_map.html")


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Utiliser la <b>racine carrée</b> de l'effectif (plutôt que l'effectif brut) pour dimensionner un symbole ponctuel est une convention cartographique classique : la perception visuelle d'une <i>surface</i> (celle du cercle) croît plus vite que sa valeur sous-jacente si l'on ne corrige pas ainsi — sans quoi Shanghai (622 étudiants) écraserait visuellement toutes les autres villes.
</div>

## Calculer des distances réelles

Pour mesurer une distance réelle entre deux points à la surface de la Terre (et non une distance « à vol d'oiseau » naïve sur des degrés de latitude/longitude, qui serait fausse), on utilise la **distance géodésique**, fournie par `geopy` :

In [ ]:
!pip install geopy
from geopy.distance import geodesic

In [ ]:
university_coords = {
    "Columbia": (40.8075, -73.9626),
    "Harvard": (42.3770, -71.1167),
    "Pennsylvania": (39.9522, -75.1932),
    "Michigan": (42.2780, -83.7382),
    "Cornell": (42.4534, -76.4735),
    "California": (37.8719, -122.2585),
    "Chicago": (41.7886, -87.5987),
    "New York University": (40.7295, -73.9965),
    "Yale": (41.3163, -72.9223),
    "Princeton": (40.3431, -74.6551),
    "Wisconsin": (43.0766, -89.4125),
    "Massachusetts Institute of Technology": (42.3601, -71.0942),
    "Illinois": (40.1020, -88.2272),
    "Stanford": (37.4275, -122.1697),
    "Purdue": (40.4237, -86.9212),
    "George Washington": (38.9002, -77.0475),
    "Northwestern": (42.0565, -87.6753),
    "Vanderbilt": (36.1447, -86.8027),
    "Georgetown": (38.9076, -77.0723),
    "Ohio State": (39.9980, -83.0305),
}

# Exemple : distance entre Columbia et Shanghai
distance_km = geodesic(university_coords["Columbia"], city_coords["Shanghai"]).kilometers
print(f"Distance Columbia – Shanghai : {distance_km:.0f} km")


On peut calculer cette distance pour **tous** les étudiants dont l'université (ou les universités) et la ville de 1936 sont géolocalisées :

In [ ]:
def calculer_distance(row):
    if row["University"] in university_coords and row["City"] in city_coords:
        return geodesic(university_coords[row["University"]], city_coords[row["City"]]).kilometers
    return np.nan       # signale les valeurs manquantes

auc["distance_km"] = auc.apply(calculer_distance, axis=1)

print(f"{auc['distance_km'].notna().sum()} trajectoires d'étudiants sur {len(auc)} ont une distance calculable")
auc["distance_km"].describe()


Les valeurs les plus **faibles** de `distance_km` correspondent aux étudiants d'origine américaine ou aux étudiants chinois qui ne sont pas repartis en Chine (installés à New York ou Pasadena) :

In [ ]:
auc.dropna(subset=["distance_km"]).sort_values("distance_km")[["Name_full", "University", "City", "distance_km"]].head(5)


## 🧪 À vous de jouer — Exercice 6

1. Quelle est la distance moyenne parcourue par les étudiants entre leur université et leur ville d'installation en 1936 ?
2. Tracez un histogramme de `distance_km`.
3. **Bonus** : ajoutez un marqueur sur `m_cities` pour une ville de votre choix.


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir la solution</summary>

```python
# Question 1
auc["distance_km"].mean()

# Question 2
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(data=auc, x="distance_km", bins=30, ax=ax)
ax.set_title("Distance parcourue entre université et ville d'installation (1936)")
ax.set_xlabel("Distance (km)")
plt.show()

# Question 3 

folium.Marker(
    location=[31.2304, 121.4737],
    popup="Shanghai",
    tooltip="Shanghai"
).add_to(m_cities)

```
</details>


# 7. Combiner réseaux et cartes : le réseau université → ville d'installation

Jusqu'ici, nous avons dessiné nos réseaux avec des dispositions **abstraites** (`spring_layout`), qui n'ont pas de signification géographique. Construisons maintenant un **nouveau** réseau biparti, distinct de celui des sections 2 et 3 : université (où l'on a étudié) ↔ ville (où l'on s'est réinstallé). Contrairement aux employeurs (des noms d'institutions, difficiles à géolocaliser un par un), les universités et les villes sont **toutes deux directement géolocalisables** avec les tables de coordonnées déjà construites — ce réseau se prête donc naturellement à une carte de flux.

## Construire le réseau université → ville

In [ ]:
links_uc = auc[["Name_full", "University", "City"]].drop_duplicates().dropna(subset=["University", "City"])
edges_uc = links_uc.groupby(["University", "City"]).size().reset_index(name="weight").sort_values("weight", ascending=False)

print(f"{len(edges_uc)} liens uniques université-ville")
edges_uc.head(10)


## Ne conserver que les liens entièrement géolocalisés

Comme pour toute donnée géocodée en contexte réel, la couverture ne sera **pas complète** : seule une partie des universités présentes dans `edges_uc` figure dans notre table de coordonnées (les plus fréquentées ; voir section 6). On filtre en le documentant explicitement, plutôt qu'en l'ignorant silencieusement :

In [ ]:
edges_geo = edges_uc[edges_uc["University"].isin(university_coords) & edges_uc["City"].isin(city_coords)]

print(f"{len(edges_geo)} liens géolocalisés sur {len(edges_uc)} au total")
print(f"{edges_geo['weight'].sum()} trajectoires étudiantes représentées sur {edges_uc['weight'].sum()} au total ({edges_geo['weight'].sum() / edges_uc['weight'].sum():.0%})")


## Carte statique du réseau géoréférencé

On dessine chaque lien comme un segment reliant l'université à la ville, avec une épaisseur proportionnelle au nombre d'étudiants sur cette route :

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))

for _, row in edges_geo.iterrows():
    lat1, lon1 = university_coords[row["University"]]
    lat2, lon2 = city_coords[row["City"]]
    ax.plot([lon1, lon2], [lat1, lat2], color="gray", alpha=0.35, linewidth=row["weight"] * 0.25, zorder=1)

for u in edges_geo["University"].unique():
    lat, lon = university_coords[u]
    total = edges_geo[edges_geo["University"] == u]["weight"].sum()
    ax.scatter(lon, lat, s=total * 6 + 30, color="#4C72B0", edgecolor="white", zorder=2)
    ax.annotate(u, (lon, lat), fontsize=6, xytext=(3, 3), textcoords="offset points")

for c in edges_geo["City"].unique():
    lat, lon = city_coords[c]
    total = edges_geo[edges_geo["City"] == c]["weight"].sum()
    ax.scatter(lon, lat, s=total * 2 + 30, color="#C44E52", edgecolor="white", zorder=2, marker="^")
    ax.annotate(c, (lon, lat), fontsize=7, xytext=(3, -8), textcoords="offset points")

ax.set_title("Réseau université (bleu) → ville d'installation en 1936 (rouge)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.show()


<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 Contrairement à un <code>spring_layout</code>, la disposition ici est <b>contrainte par la réalité géographique</b> : on obtient, de fait, une véritable <b>carte de flux migratoires</b> — des campus américains vers les villes chinoises (essentiellement Shanghai) où s'est reconstituée, en 1936, la communauté des anciens étudiants.
</div>

## Version interactive avec folium

On reproduit la même carte en interactif, avec des courbes et des marqueurs proportionnels :

In [ ]:
m_flows = folium.Map(location=[30, 40], zoom_start=2, tiles="OpenStreetMap")

for _, row in edges_geo.iterrows():
    folium.PolyLine(
        locations=[university_coords[row["University"]], city_coords[row["City"]]],
        color="gray", weight=row["weight"] * 0.5, opacity=0.5
    ).add_to(m_flows)

for u in edges_geo["University"].unique():
    total = edges_geo[edges_geo["University"] == u]["weight"].sum()
    folium.CircleMarker(
        location=university_coords[u], radius=total * 0.8 + 4,
        popup=f"{u} ({total} étudiants installés ensuite)",
        color="#4C72B0", fill=True, fill_opacity=0.8
    ).add_to(m_flows)

for c in edges_geo["City"].unique():
    total = edges_geo[edges_geo["City"] == c]["weight"].sum()
    folium.CircleMarker(
        location=city_coords[c], radius=total ** 0.5 * 2 + 4,
        popup=f"{c} ({total} étudiants accueillis)",
        color="#C44E52", fill=True, fill_opacity=0.8
    ).add_to(m_flows)

m_flows

# m_flows.save("m_flows.html")


In [ ]:
m_flows = folium.Map(
    location=[30, 40],
    zoom_start=2,
    tiles="OpenStreetMap"
)

# Liens université → ville
for _, row in edges_geo.iterrows():

    folium.PolyLine(
        locations=[
            university_coords[row["University"]],
            city_coords[row["City"]]
        ],
        color="gray",
        weight=row["weight"] * 0.5,
        opacity=0.5,
        tooltip=(
            f"From {row['University']} to {row['City']} "
            f"— {row['weight']} trajectoire(s)"
        )
    ).add_to(m_flows)


# Universités
for u in edges_geo["University"].unique():

    total = edges_geo[
        edges_geo["University"] == u
    ]["weight"].sum()

    folium.CircleMarker(
        location=university_coords[u],
        radius=total * 0.8 + 4,
        popup=f"{u} ({total} étudiant(s) accueilli(s))",
        color="#4C72B0",
        fill=True,
        fill_opacity=0.8
    ).add_to(m_flows)


# Villes
for c in edges_geo["City"].unique():

    total = edges_geo[
        edges_geo["City"] == c
    ]["weight"].sum()

    folium.CircleMarker(
        location=city_coords[c],
        radius=total ** 0.5 * 2 + 4,
        popup=f"{c} ({total} étudiant(s) installé(s)",
        color="#C44E52",
        fill=True,
        fill_opacity=0.8
    ).add_to(m_flows)

m_flows

# m_flows.save("m_flows.html")

## 🧪 À vous de jouer — Exercice 7

1. Quelle université envoie, proportionnellement à son effectif total, la plus grande part de ses diplômés **hors de Shanghai** ?
2. Quelle est la ville (hors Shanghai) la plus « diversifiée » en termes d'universités d'origine de ses étudiants ?
3. **Discussion (à l'oral, pas de code)** : cette carte de flux repose sur une couverture partielle des universités et villes (voir plus haut). En quoi cela pourrait-il biaiser l'image que l'on se fait de la géographie de ces migrations de retour ?


In [ ]:
# 🧪 Essayez ici




<details>
<summary>▶️ Voir une piste de solution</summary>

```python
# Question 1
non_shanghai = edges_geo[edges_geo["City"] != "Shanghai"]
totals_by_univ = edges_geo.groupby("University")["weight"].sum()
non_shanghai_by_univ = non_shanghai.groupby("University")["weight"].sum()
share = (non_shanghai_by_univ / totals_by_univ).dropna().sort_values(ascending=False)
share.head(5)

# Question 2
edges_geo[edges_geo["City"] != "Shanghai"].groupby("City")["University"].nunique().sort_values(ascending=False)
```
</details>


# 8. Mini-projet de synthèse

Combinons l'ensemble des méthodes vues aujourd'hui sur une **petite investigation en autonomie** (25-30 min). Travaillez seul·e ou en binôme.

## 🧪 Consignes

Choisissez **l'une** des trois pistes suivantes (ou deux, si le temps le permet) :

- ****A. Approfondir l'analyse de réseau** : concentrez-vous sur un sous-ensemble pertinent du réseau université–employeur (par exemple, un champ disciplinaire, une période, ou une nationalité) et étudiez sa structure relationnelle (nombre de noeuds/liens, centralités, communautés...).
- **B. Approfondir la cartographie** : cartographiez la variable `Sector_1` (secteur d'emploi en 1936) croisée avec `City`, ou bien construisez une carte choroplèthe de la **durée moyenne des études** (`Year_end - Year_start`) par État de formation. Produisez une carte statique **et** interactive, avec un titre et une légende clairs.
- **C. Approfondir le réseau géoréférencé** : reprenez le réseau université → ville de la section 7, mais restreignez-le aux seuls étudiants de nationalité `"Chinese"` (ou `"Western"`). Le réseau de flux diffère-t-il selon la nationalité ?

Concluez par quelques lignes de synthèse : qu'apprenez-vous sur la structure relationnelle de votre terrain d'étude, que la seule lecture du tableau de données n'aurait pas permis de voir ? Qu'apporte la carte (ou le réseau géoréférencé) que la version purement statistique ou purement relationnelle ne montrait pas ? 


In [ ]:
# 🧪 Votre code ici — utilisez autant de cellules que nécessaire





<details>
<summary>▶️ Voir un exemple de démarche (piste A, durée moyenne des études par État)</summary>

```python
auc["duree"] = auc["Year_end"] - auc["Year_start"]
duree_par_etat = auc.groupby("State")["duree"].mean().reset_index()

us_duree = us_states.merge(duree_par_etat, left_on="name", right_on="State", how="left")

fig, ax = plt.subplots(figsize=(12, 7))
us_duree.plot(column="duree", ax=ax, cmap="viridis", legend=True, edgecolor="#8a8060", linewidth=0.4,
              missing_kwds={"color": "#EFEAD9"})
ax.set_title("Durée moyenne des études, par État de formation")
ax.axis("off")
plt.show()
```
</details>


# Aide-mémoire

## Glossaire — réseaux

| Terme | Description |
|---|---|
| Nœud / Lien (*node* / *edge*) | Une entité, une relation entre deux entités. |
| Réseau biparti | Deux types de nœuds distincts, liens uniquement entre les deux types. |
| Projection | Transformation d'un réseau biparti en réseau à un seul type de nœuds. |
| Centralité (degré, intermédiarité, proximité, vecteur propre) | Mesures de l'importance relative d'un nœud. |
| Communauté / Modularité | Sous-groupe dense de nœuds / mesure de la qualité d'un tel découpage. |

## Glossaire — analyse spatiale

| Terme | Description |
|---|---|
| **GeoDataFrame** | Un DataFrame pandas doté d'une colonne `geometry`. |
| **Géométrie** (Point, LineString, Polygon) | La forme spatiale d'une observation. |
| **CRS** (*Coordinate Reference System*) | Le système reliant des coordonnées numériques à des positions réelles (ex. EPSG:4326 = latitude/longitude en degrés). |
| **Reprojection** | Conversion d'un CRS à un autre (`.to_crs()`), nécessaire avant un calcul de distance/surface rigoureux. |
| **Carte choroplèthe** | Carte où chaque unité géographique est colorée selon la valeur d'une variable. |
| **Carte de flux** | Carte représentant des déplacements ou relations entre lieux par des lignes reliant des points. |
| **Géocodage** | Conversion d'un nom de lieu en coordonnées (latitude/longitude). |
| **Distance géodésique** | Distance réelle à la surface de la Terre entre deux points (par opposition à une distance naïve en degrés). |

## Index des fonctions

| Fonction | Bibliothèque | Rôle |
|---|---|---|
| `nx.Graph()`, `.add_node()`, `.add_edge()` | networkx | Créer un réseau |
| `bipartite.weighted_projected_graph()` | networkx.algorithms.bipartite | Projeter un réseau biparti |
| `nx.degree_centrality()` / `betweenness_centrality()` / `closeness_centrality()` / `eigenvector_centrality()` | networkx | Mesures de centralité |
| `nx.community.louvain_communities()` / `.modularity()` | networkx | Détection de communautés |
| `gpd.read_file()` | geopandas | Charger un fichier spatial (GeoJSON, Shapefile...) |
| `gdf.crs` / `gdf.to_crs()` | geopandas | Consulter / reprojeter le système de coordonnées |
| `gdf.plot(column=...)` | geopandas | Tracer une carte statique (choroplèthe si `column=`) |
| `folium.Map()` | folium | Créer une carte interactive |
| `folium.Choropleth()` | folium | Carte choroplèthe interactive |
| `folium.Marker()` / `.CircleMarker()` | folium | Ajouter un point sur une carte interactive |
| `folium.PolyLine()` | folium | Tracer une ligne (ex. un lien de réseau) sur une carte |
| `geopy.distance.geodesic()` | geopy | Distance réelle entre deux points (lat, lon) |
| `geopy.geocoders.Nominatim` | geopy | Géocodage (nom de lieu → coordonnées) |

## Où trouver de l'aide ?

1. **Documentation officielle** : [NetworkX](https://networkx.org/documentation/stable/), [geopandas](https://geopandas.org/), [folium](https://python-visualization.github.io/folium/), [geopy](https://geopy.readthedocs.io/).
2. **The Programming Historian** : tutoriels sur l'analyse de réseau et la cartographie historique en SHS ([programminghistorian.org](https://programminghistorian.org/)).
3. **Gephi** ([gephi.org](https://gephi.org/)) et **QGIS** ([qgis.org](https://qgis.org/)) : logiciels libres à interface graphique, bons compléments à Python pour l'exploration visuelle sans code (export possible avec `nx.write_gexf()` pour Gephi, ou tout format spatial standard pour QGIS).
4. **Stack Overflow, ChatGPT, Claude** : comme toujours, donnez le maximum de contexte (message d'erreur complet, structure des données, bibliothèque utilisée).

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
💡 <b>Pour prolonger cette séance</b> : réseaux et cartes se combinent dans de nombreux contextes de recherche — réseaux de migration, circulation de correspondances, diffusion d'idées ou de maladies, structuration régionale d'une administration... Toute donnée comportant à la fois une dimension relationnelle et une dimension géographique peut être explorée avec les deux approches vues aujourd'hui, séparément ou (comme en section 7) ensemble.
</div>


---

# Annexe : réseau de co-occurrence à partir d'un corpus de texte

Cette annexe reprend une méthode que nous n'avons pas eu le temps de voir en séance : construire un réseau **à partir d'un corpus de texte** plutôt que d'un tableau de données — en l'occurrence, un réseau de co-occurrence entre entités nommées extraites du corpus de presse olympique (Séance 2). Elle applique exactement la même logique que les sections 2 et 3 (construction, filtrage, centralité, communautés), simplement appliquée à un objet de départ différent. Une bonne lecture pour prolonger la séance chez vous.

## Recharger et préparer le corpus


In [ ]:
import spacy
import itertools
from collections import Counter

corpus = pd.read_csv("data/olympic_corpus.csv")
corpus = corpus.drop(columns=["Unnamed: 0"])
corpus["Date"] = pd.to_datetime(corpus["Date"], format="%Y%m%d", errors="coerce")
corpus["Year"] = corpus["Date"].dt.year

def clean_text(text):
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", text).strip()

corpus["Text_clean"] = corpus["Text"].apply(clean_text)

editorial_types = ["Feature/Article", "General News", "Editorial/Opinion", "Review"]
articles = corpus[corpus["category_clean"].isin(editorial_types)].copy()
articles["n_words"] = articles["Text_clean"].str.split().str.len()

sample = articles[articles["n_words"] >= 50].sample(n=500, random_state=42).reset_index(drop=True)
len(sample)


## Étape 1 : extraire les entités nommées de chaque article

On réutilise spaCy (Séance 2) pour extraire, pour chaque article, l'ensemble des entités de type `PERSON`, `GPE` (lieu) et `ORG` (organisation) :

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["lemmatizer"])

doc_entities = {}   # DocId -> ensemble d'entités mentionnées dans ce document

for doc_id, doc in zip(sample["DocId"], nlp.pipe(sample["Text_clean"], batch_size=50)):
    ents = set()
    for ent in doc.ents:
        if ent.label_ in ("PERSON", "GPE", "ORG") and len(ent.text) > 2:
            ents.add(ent.text.strip())
    doc_entities[doc_id] = ents

# Exemple pour le premier document
list(doc_entities.values())[0]


## Étape 2 : construire les liens de co-occurrence

Pour chaque article, on relie **toutes les paires possibles** d'entités qui y apparaissent ensemble (`itertools.combinations`), et l'on cumule le poids de chaque paire sur l'ensemble du corpus :

In [ ]:
node_counter = Counter()   # nombre d'articles mentionnant chaque entité
edge_counter = Counter()   # nombre d'articles où chaque paire d'entités co-apparaît

for ents in doc_entities.values():
    for e in ents:
        node_counter[e] += 1
    for a, b in itertools.combinations(sorted(ents), 2):
        edge_counter[(a, b)] += 1

print(f"{len(node_counter)} entités distinctes, {len(edge_counter)} paires de co-occurrence")

<div class="alert alert-success" role="alert" style="background-color:#d4edda;padding:10px;border-radius:5px;">
<b>Explications</b>

- <code>itertools.combinations(sorted(ents), 2)</code> génère toutes les paires possibles d'entités présentes dans un même document (sans répétition, ni paire d'un nœud avec lui-même). Trier les entités avant de générer les paires assure que la paire (A, B) est toujours produite dans le même ordre que (A, B), et jamais aussi comme (B, A) — ce qui éviterait de compter deux fois le même lien.
- <code>Counter</code> (du module <code>collections</code>) est une structure très efficace pour compter des occurrences — un dictionnaire spécialisé qui retourne 0 par défaut pour une clé absente.
</div>

## Étape 3 : filtrer et construire le graphe

Avec plusieurs milliers d'entités distinctes, le réseau complet est totalement illisible (et long à calculer). On se concentre sur les entités les plus fréquentes, et sur les liens les plus robustes :

In [ ]:
TOP_N = 60
MIN_WEIGHT = 2

top_entities = {e for e, _ in node_counter.most_common(TOP_N)}

G_ent = nx.Graph()
for e in top_entities:
    G_ent.add_node(e, weight=node_counter[e])
for (a, b), w in edge_counter.items():
    if a in top_entities and b in top_entities and w >= MIN_WEIGHT:
        G_ent.add_edge(a, b, weight=w)

print(f"{G_ent.number_of_nodes()} nœuds, {G_ent.number_of_edges()} liens")

communities_ent = nx.community.louvain_communities(G_ent, weight="weight", seed=42)
print(f"{len(communities_ent)} communautés détectées")


## Étape 4 : visualiser le réseau (statique)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

pos = nx.spring_layout(G_ent, seed=42, k=0.5)
node_sizes = [G_ent.nodes[n]["weight"] * 15 for n in G_ent.nodes()]
edge_widths = [G_ent[u][v]["weight"] * 0.3 for u, v in G_ent.edges()]

nx.draw_networkx_nodes(G_ent, pos, node_size=node_sizes, node_color="#C44E52", alpha=0.8, ax=ax)
nx.draw_networkx_edges(G_ent, pos, width=edge_widths, edge_color="lightgray", ax=ax)
nx.draw_networkx_labels(G_ent, pos, font_size=7, ax=ax)

ax.set_title(f"Entity co-occurrence network (top {TOP_N} entities, min. {MIN_WEIGHT} shared articles)")
ax.axis("off")
plt.show()

## Etape 5 : visualisation interactive avec pyvis

In [ ]:
from pyvis.network import Network

net = Network(height="700px", width="100%", notebook=True, cdn_resources="in_line", bgcolor="white")
net.from_nx(G_ent)

# Active la simulation physique interactive (on peut glisser les nœuds à la souris)
net.show_buttons(filter_=["physics"])

net.show("entity_network.html")


## 🧪 À vous de jouer — Exercice annexe

1. Quelles sont les 10 entités les plus fréquemment mentionnées (`node_counter.most_common(10)`) ?
2. Recommencez la construction du réseau avec `TOP_N = 40` et `MIN_WEIGHT = 3` : le réseau est-il plus lisible ? Combien de nœuds/liens obtenez-vous ?
3. Quelle paire d'entités co-apparaît dans le plus grand nombre d'articles (indice : `edge_counter.most_common(10)`) ?
4. **Bonus** : reconstruisez le réseau de co-occurrence en vous limitant aux articles publiés entre 1935 et 1936 (contexte des Jeux de Berlin). Le réseau obtenu met-il en avant des entités différentes ?


In [ ]:
# 🧪 Essayez ici

<details>
<summary>▶️ Voir la solution</summary>

```python
# Question 1
node_counter.most_common(10)

# Question 2
top_entities_40 = {e for e, _ in node_counter.most_common(40)}
G_ent_2 = nx.Graph()
for e in top_entities_40:
    G_ent_2.add_node(e, weight=node_counter[e])
for (a, b), w in edge_counter.items():
    if a in top_entities_40 and b in top_entities_40 and w >= 3:
        G_ent_2.add_edge(a, b, weight=w)
print(G_ent_2.number_of_nodes(), G_ent_2.number_of_edges())

# Question 3
edge_counter.most_common(10)

# Question 4 (bonus)
sample_berlin = articles[(articles["Year"] >= 1935) & (articles["Year"] <= 1936) & (articles["n_words"] >= 50)]
doc_entities_berlin = {}
for doc_id, doc in zip(sample_berlin["DocId"], nlp.pipe(sample_berlin["Text_clean"], batch_size=50)):
    ents = {ent.text.strip() for ent in doc.ents if ent.label_ in ("PERSON", "GPE", "ORG") and len(ent.text) > 2}
    doc_entities_berlin[doc_id] = ents
# ... puis reconstruire node_counter / edge_counter / G_ent comme ci-dessus
```
</details>